In [1]:
import numpy as np
import cv2
import pickle
import os

# Load Phase 2 results
with open('phase2_results.pkl', 'rb') as f:
    p2 = pickle.load(f)

img = p2['img']
height, width = p2['height'], p2['width']
region_map = p2['region_map']
special_regions = p2['special_regions']

output_dir = "comic/phase3_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Phase 2 data loaded")
print(f"Special regions (text+edge): {len(special_regions)}")

# Get gradient candidate regions (not special)
gradient_candidates = set(np.unique(region_map[region_map >= 0])) - special_regions
print(f"Gradient candidate regions: {len(gradient_candidates)}")

Phase 2 data loaded
Special regions (text+edge): 11387
Gradient candidate regions: 2271


In [2]:
def detect_gradient(rid, rmap, img, mag_threshold=10):
    """
    Detect if a region is gradient using Sobel method
    """
    region_mask = (rmap == rid)
    region_pixels = np.argwhere(region_mask)
    
    if len(region_pixels) < 10:
        return 'flat'
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(float)
    Gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    Gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    
    magnitude = np.sqrt(Gx**2 + Gy**2)
    theta = np.arctan2(Gy, Gx)
    
    region_mag = magnitude[region_mask]
    region_theta = theta[region_mask]
    
    if np.mean(region_mag) < mag_threshold:
        return 'flat'
    
    theta_deg = (np.degrees(region_theta) + 180) % 360
    direction_bin = (theta_deg / 45).astype(int) % 8
    hist, _ = np.histogram(direction_bin, bins=np.arange(0, 9))
    
    if np.max(hist) < len(region_pixels) * 0.3:
        return 'flat'
    
    main_direction = np.argmax(hist)
    region_pixels_array = np.array(region_pixels)
    
    angle_rad = np.radians(main_direction * 45)
    dir_vec = np.array([np.cos(angle_rad), np.sin(angle_rad)])
    projections = region_pixels_array @ dir_vec
    
    min_idx = np.argmin(projections)
    max_idx = np.argmax(projections)
    
    start_color = img[region_pixels_array[min_idx][0], region_pixels_array[min_idx][1]].astype(float)
    end_color = img[region_pixels_array[max_idx][0], region_pixels_array[max_idx][1]].astype(float)
    
    return ('gradient', main_direction, start_color, end_color)

# Classify regions
gradient_regions = {}
flat_regions = []

for rid in gradient_candidates:
    result = detect_gradient(rid, region_map, img)
    
    if result == 'flat':
        flat_regions.append(rid)
    else:
        gradient_regions[rid] = {
            'direction': result[1],
            'start_color': result[2],
            'end_color': result[3]
        }

print(f"\nGradient detection results:")
print(f"  Gradient regions: {len(gradient_regions)}")
print(f"  Flat regions: {len(flat_regions)}")


Gradient detection results:
  Gradient regions: 315
  Flat regions: 1956


In [3]:
def get_direct_neighbors(rid, rmap):
    region_mask = (rmap == rid)
    neighbors = set()
    for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        shifted = np.roll(np.roll(region_mask, dy, axis=0), dx, axis=1)
        neighbor_ids = np.unique(rmap[shifted & ~region_mask])
        neighbors.update(neighbor_ids)
    neighbors.discard(rid)
    neighbors.discard(-1)
    return list(neighbors)

def directions_compatible(dir1, dir2, tolerance=1):
    diff = abs(dir1 - dir2)
    diff = min(diff, 8 - diff)
    return diff <= tolerance

# Union-Find
parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

# Find adjacent same-direction gradients
for rid in gradient_regions.keys():
    neighbors = get_direct_neighbors(rid, region_map)
    for neighbor_rid in neighbors:
        if neighbor_rid not in gradient_regions:
            continue
        if directions_compatible(gradient_regions[rid]['direction'], 
                                gradient_regions[neighbor_rid]['direction']):
            union(rid, neighbor_rid)

# Group by root
groups = {}
for rid in gradient_regions.keys():
    root = find(rid)
    if root not in groups:
        groups[root] = []
    groups[root].append(rid)

print(f"After merging adjacent gradients: {len(groups)} groups")

# Update gradient_regions
merged_gradient_regions = {}

for group_root, group_rids in groups.items():
    merged_rid = group_root
    
    group_mask = np.zeros((height, width), dtype=bool)
    for rid in group_rids:
        group_mask |= (region_map == rid)
    
    group_pixels = np.argwhere(group_mask)
    
    if len(group_pixels) == 0:
        continue
    
    direction = gradient_regions[group_root]['direction']
    
    angle_rad = np.radians(direction * 45)
    dir_vec = np.array([np.cos(angle_rad), np.sin(angle_rad)])
    projections = group_pixels @ dir_vec
    
    min_idx = np.argmin(projections)
    max_idx = np.argmax(projections)
    
    start_color = img[group_pixels[min_idx][0], group_pixels[min_idx][1]].astype(float)
    end_color = img[group_pixels[max_idx][0], group_pixels[max_idx][1]].astype(float)
    
    merged_gradient_regions[merged_rid] = {
        'direction': direction,
        'start_color': start_color,
        'end_color': end_color,
        'merged_from': group_rids
    }

gradient_regions = merged_gradient_regions
print(f"Merged gradient regions: {len(gradient_regions)}")

After merging adjacent gradients: 226 groups
Merged gradient regions: 226


In [4]:
phase3_results = {
    'img': img,
    'height': height,
    'width': width,
    'region_map': region_map,
    'special_regions': special_regions,
    'gradient_regions': gradient_regions,
    'flat_regions': flat_regions
}

with open('phase3_results.pkl', 'wb') as f:
    pickle.dump(phase3_results, f)

print("Phase 3 results saved to phase3_results.pkl")
print(f"Summary:")
print(f"  Special regions (text+edge): {len(special_regions)}")
print(f"  Gradient regions: {len(gradient_regions)}")
print(f"  Flat regions: {len(flat_regions)}")

Phase 3 results saved to phase3_results.pkl
Summary:
  Special regions (text+edge): 11387
  Gradient regions: 226
  Flat regions: 1956


In [5]:
# Visualize the classification results

# Create visualization
vis = img.copy().astype(float)

# Mark special regions (text+edge) - Red
for rid in special_regions:
    mask = (region_map == rid)
    vis[mask] = [255, 0, 0]  # Red

# Mark gradient regions - Green
for rid in gradient_regions.keys():
    mask = (region_map == rid)
    vis[mask] = [0, 255, 0]  # Green

# Mark flat regions - Blue (just a few for reference)
flat_count = 0
for rid in flat_regions:
    if flat_count < 100:  # Just show first 100 for visibility
        mask = (region_map == rid)
        vis[mask] = [0, 0, 255]  # Blue
        flat_count += 1

# Save visualization
vis_path = os.path.join(output_dir, "01_phase3_classification.jpg")
cv2.imwrite(vis_path, cv2.cvtColor(vis.astype(np.uint8), cv2.COLOR_RGB2BGR))
print(f"Classification visualization saved: {vis_path}")

print(f"\nColor legend:")
print(f"  Red = Special regions (text+edge)")
print(f"  Green = Gradient regions")
print(f"  Blue = Flat regions (sample)")

Classification visualization saved: comic/phase3_output\01_phase3_classification.jpg

Color legend:
  Red = Special regions (text+edge)
  Green = Gradient regions
  Blue = Flat regions (sample)
